In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Ecosystems.reaction_segment_generation import RSG

In [2]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

Device is connected


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=36,
    y_offset=8,
    x_min=27,
    x_max=127,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=201,
    y_offset=39,
    x_min=157,
    x_max=247,
    y_min=2,
    y_max=292
) 

# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=pump, syringe_diameter=4.606)

In [6]:
g.home()

All axes homed successfully and Z is within safe limits


In [12]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(39.0))

In [44]:

rate = 0.05          # mL/min
volume = 10           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRINF


In [45]:
pump.start()

Sent: @00*RUN 1


'00I'

In [20]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(39.0))

In [80]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [81]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [74]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [75]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [58]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [59]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [76]:
rsg.pickup_from_vial("rack1", 1, 40)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL40
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [77]:
g.go_to_vial("rack1", 19)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(52.5675))

In [78]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [79]:
rsg.dispense_in_vial("rack1", 19, 65)

#Note - analyte vol + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL,
# this 10uL is reintroduced before the wash sequence)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL65
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP
